# Tidy Cloud Training

Run all cells. Checkpoints save to `MyDrive/Tidy/models` every 100 episodes; resume is automatic.

- Runtime → Change runtime type → **CPU** (physics-bound; GPU doesn't help).
- Session dies whenever Colab kills it — nothing is lost, just re-run all cells.
- Code comes from Drive: edit locally, run `./scripts/sync_code.sh`, then re-run cell 3 + 4 here. No git commit needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/Tidy
!mkdir -p /content/Tidy
!cp -a /content/drive/MyDrive/Tidy/code/. /content/Tidy/
%cd /content/Tidy

In [ ]:
import importlib.util
missing = [p for p in ("torch", "pybullet", "gymnasium", "numpy", "stable_baselines3")
           if importlib.util.find_spec(p) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    print("All packages already present, skipping install.")

In [ ]:
import os
os.environ['TIDY_MODELS_DIR'] = '/content/drive/MyDrive/Tidy/models'
!python train.py --minutes 600 --episodes 200 --alpha 0.2

## SB3 SAC sanity check (gripper probe)

Run **instead of** the `train.py` cell above. Same env and reward, textbook SAC from
Stable-Baselines3 — the control for `rl/sac.py`.

`--curriculum 0.8` starts 80% of episodes hovering just above the cube, so a working
implementation should bank `grasp` reward within a few dozen episodes. `grasp` pinned
at 0.00 while `reach` climbs means the env/reward is the suspect, not the hand-rolled SAC.

Everything is printed per 10 episodes: mean return, successes, and the per-component
breakdown. Paste the log back when the session dies.

In [ ]:
!python train_sb3.py --minutes 180 --out /content/drive/MyDrive/Tidy/models/sb3_sanity.zip